In [1]:
!pip install torch_geometric --quiet

# kaggle pytorch version and cuda version:
# pytorch version 2.10.0
# cuda version 12.8
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.10.0+cu128.html --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 83.7 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 99.9 MB/s eta 0:00:00:00:01


In [2]:
import torch
from torch import nn, Tensor
import torch.nn.functional as F

from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
    classification_report
)

from sklearn.utils.class_weight import compute_class_weight

In [ ]:
RANDOM_STATE = 42

# target variable
TARGET = 'Traffic'
TO_DROP = ['Target', 'Traffic', 'StartTime', 'LastTime', 'SrcAddr', 'DstAddr', 'sIpId', 'dIpId']

# Path on kaggle: '/kaggle/input/datasets/jiahao1510/wustl-2021/wustl_iiot_2021.csv'
df = pd.read_csv('../data/wustl_iiot_2021.csv')

In [4]:
df['Dport'].value_counts()

Dport
502      1103533
80         52935
0          11114
8080        7221
1740        4812
          ...   
55127          1
53549          1
63261          1
250            1
39951          1
Name: count, Length: 7781, dtype: int64

In [5]:
df.columns

Index(['StartTime', 'LastTime', 'SrcAddr', 'DstAddr', 'Mean', 'Sport', 'Dport',
       'SrcPkts', 'DstPkts', 'TotPkts', 'DstBytes', 'SrcBytes', 'TotBytes',
       'SrcLoad', 'DstLoad', 'Load', 'SrcRate', 'DstRate', 'Rate', 'SrcLoss',
       'DstLoss', 'Loss', 'pLoss', 'SrcJitter', 'DstJitter', 'SIntPkt',
       'DIntPkt', 'Proto', 'Dur', 'TcpRtt', 'IdleTime', 'Sum', 'Min', 'Max',
       'sDSb', 'sTtl', 'dTtl', 'sIpId', 'dIpId', 'SAppBytes', 'DAppBytes',
       'TotAppByte', 'SynAck', 'RunTime', 'sTos', 'SrcJitAct', 'DstJitAct',
       'Traffic', 'Target'],
      dtype='object')

In [6]:
all_ip_addresses = set(df['SrcAddr']).union(set(df['DstAddr']))

ip_addr_mapping = {ip_address: i for i, ip_address in enumerate(all_ip_addresses)}

In [7]:
ip_addr_mapping

{'5099': 0,
 '66589': 1,
 '66448': 2,
 '4886': 3,
 '66540': 4,
 '17610': 5,
 '4926': 6,
 '66715': 7,
 '209.240.235.92': 8,
 '53329': 9,
 'ff02::1:3': 10,
 '66385': 11,
 '16274': 12,
 '15046': 13,
 '5005': 14,
 '5092': 15,
 '67263': 16,
 '65972': 17,
 '5097': 18,
 '5114': 19,
 '5108': 20,
 '65771': 21,
 '192.168.0.4': 22,
 '66343': 23,
 '4504': 24,
 '5081': 25,
 '5087': 26,
 '5085': 27,
 '5098': 28,
 'fe80::dacb:8aff:fe08:ed2a': 29,
 '4878': 30,
 '5091': 31,
 '192.168.0.255': 32,
 '66335': 33,
 '30515': 34,
 '82621': 35,
 '4925': 36,
 'ff02::16': 37,
 '30741': 38,
 '66401': 39,
 '192.168.0.20': 40,
 '5103': 41,
 '224.0.0.251': 42,
 'ff02::1': 43,
 '5084': 44,
 '17600': 45,
 '4871': 46,
 '65886': 47,
 '77980': 48,
 '4951': 49,
 '49.48.134.64': 50,
 '192.168.0.2': 51,
 '75281': 52,
 '72918': 53,
 '73748': 54,
 '66330': 55,
 'fe80::9bc:3b2b:78d3:855c': 56,
 '66414': 57,
 '5086': 58,
 '30625': 59,
 '90144': 60,
 '5579': 61,
 '66267': 62,
 '4873': 63,
 '5100': 64,
 '4989': 65,
 '17614': 66,


In [8]:
df.columns

Index(['StartTime', 'LastTime', 'SrcAddr', 'DstAddr', 'Mean', 'Sport', 'Dport',
       'SrcPkts', 'DstPkts', 'TotPkts', 'DstBytes', 'SrcBytes', 'TotBytes',
       'SrcLoad', 'DstLoad', 'Load', 'SrcRate', 'DstRate', 'Rate', 'SrcLoss',
       'DstLoss', 'Loss', 'pLoss', 'SrcJitter', 'DstJitter', 'SIntPkt',
       'DIntPkt', 'Proto', 'Dur', 'TcpRtt', 'IdleTime', 'Sum', 'Min', 'Max',
       'sDSb', 'sTtl', 'dTtl', 'sIpId', 'dIpId', 'SAppBytes', 'DAppBytes',
       'TotAppByte', 'SynAck', 'RunTime', 'sTos', 'SrcJitAct', 'DstJitAct',
       'Traffic', 'Target'],
      dtype='object')

In [9]:
# private ip addresses start with 192.168
is_private_ip = {node_idx: 1 if ip.startswith('192.168') else 0 for ip, node_idx in ip_addr_mapping.items()}
# node id: is_private_ip
is_private_ip

{0: 0,
 1: 0,
 2: 0,
 3: 0,
 4: 0,
 5: 0,
 6: 0,
 7: 0,
 8: 0,
 9: 0,
 10: 0,
 11: 0,
 12: 0,
 13: 0,
 14: 0,
 15: 0,
 16: 0,
 17: 0,
 18: 0,
 19: 0,
 20: 0,
 21: 0,
 22: 1,
 23: 0,
 24: 0,
 25: 0,
 26: 0,
 27: 0,
 28: 0,
 29: 0,
 30: 0,
 31: 0,
 32: 1,
 33: 0,
 34: 0,
 35: 0,
 36: 0,
 37: 0,
 38: 0,
 39: 0,
 40: 1,
 41: 0,
 42: 0,
 43: 0,
 44: 0,
 45: 0,
 46: 0,
 47: 0,
 48: 0,
 49: 0,
 50: 0,
 51: 1,
 52: 0,
 53: 0,
 54: 0,
 55: 0,
 56: 0,
 57: 0,
 58: 0,
 59: 0,
 60: 0,
 61: 0,
 62: 0,
 63: 0,
 64: 0,
 65: 0,
 66: 0,
 67: 0,
 68: 0,
 69: 0,
 70: 0,
 71: 0,
 72: 0,
 73: 0,
 74: 0,
 75: 0,
 76: 0,
 77: 0,
 78: 0,
 79: 0,
 80: 0,
 81: 0,
 82: 0,
 83: 0,
 84: 0,
 85: 0,
 86: 1,
 87: 0,
 88: 0,
 89: 0,
 90: 0,
 91: 0,
 92: 0,
 93: 0,
 94: 0,
 95: 0,
 96: 0,
 97: 0,
 98: 0,
 99: 0,
 100: 0,
 101: 0,
 102: 0,
 103: 0,
 104: 0,
 105: 0,
 106: 0,
 107: 0,
 108: 0,
 109: 0,
 110: 0,
 111: 0,
 112: 0,
 113: 0,
 114: 0,
 115: 0,
 116: 0,
 117: 0,
 118: 0,
 119: 1,
 120: 0,
 121: 0,
 122: 0,
 12

In [10]:
# encode the ip addresses into node indices
df['SrcAddr'] = df['SrcAddr'].map(ip_addr_mapping)
df['DstAddr'] = df['DstAddr'].map(ip_addr_mapping)

df = df.sort_values(by=['SrcAddr', 'DstAddr'])

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[TARGET]
)

df_train, df_val = train_test_split(
    df_train,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=df_train[TARGET]
)

encoder = LabelEncoder()
df_train[TARGET] = encoder.fit_transform(df_train[TARGET])
df_val[TARGET] = encoder.transform(df_val[TARGET])
df_test[TARGET] = encoder.transform(df_test[TARGET])

In [11]:
continuous_cols = ['SrcPkts', 'DstPkts', 'TotPkts', 'DstBytes', 'SrcBytes', 'TotBytes',
    'SrcLoad', 'DstLoad', 'Load', 'SrcRate', 'DstRate', 'Rate', 'SrcLoss',
    'DstLoss', 'Loss', 'pLoss', 'SrcJitter', 'DstJitter', 'SIntPkt', 'DIntPkt',
    'Dur', 'TcpRtt', 'IdleTime', 'Sum', 'Min', 'Max', 'SAppBytes', 'DAppBytes',
    'TotAppByte', 'RunTime', 'SrcJitAct', 'DstJitAct', 'Sport', 'Dport', 'sTtl', 'dTtl', 'SynAck']

categorical_cols = ['Proto', 'sDSb', 'sTos']

binary_cols = ['is_private_src', 'is_private_dst']

# fit on train only
scaler = StandardScaler()
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

scaler.fit(df_train[continuous_cols])
ohe.fit(df_train[categorical_cols])

ohe_col_names = ohe.get_feature_names_out(categorical_cols)

def transform_df(df, scaler, ohe, continuous_cols, categorical_cols, ohe_col_names, binary_cols):
    df = df.copy()

    # continuous -> standardized, overwrite in place
    df[continuous_cols] = scaler.transform(df[continuous_cols])

    # categorical -> one-hot, new columns appended
    ohe_array = ohe.transform(df[categorical_cols])
    ohe_df = pd.DataFrame(ohe_array, columns=ohe_col_names, index=df.index)
    df = pd.concat([df, ohe_df], axis=1)

    # drop original categorical columns now that they're one-hot encoded
    df = df.drop(columns=categorical_cols)

    return df

df_train = transform_df(df_train, scaler, ohe, continuous_cols, categorical_cols, ohe_col_names, binary_cols)
df_val = transform_df(df_val, scaler, ohe, continuous_cols, categorical_cols, ohe_col_names, binary_cols)
df_test = transform_df(df_test, scaler, ohe, continuous_cols, categorical_cols, ohe_col_names, binary_cols)

In [12]:
df_train

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,sDSb_0,sDSb_4,sDSb_31,sDSb_48,sDSb_51,sTos_0,sTos_16,sTos_126,sTos_192,sTos_207
972507,2019-08-19 15:03:05,2019-08-19 15:03:05,40,51,0,0.445750,-0.087658,-0.002955,-0.007720,-0.002975,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
879947,2019-08-19 12:43:37,2019-08-19 12:43:37,8,51,0,-2.825807,-0.215384,-0.003104,-0.015094,-0.003274,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1024301,2019-08-19 16:38:07,2019-08-19 16:38:07,40,51,0,0.801632,-0.087658,-0.002955,-0.007720,-0.002975,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
38249,2019-08-19 15:13:58,2019-08-19 15:13:58,40,51,0,0.252253,-0.087658,-0.002955,-0.007720,-0.002975,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
130341,2019-08-19 09:53:22,2019-08-19 09:53:22,40,51,0,0.894886,-0.087658,-0.002955,-0.007720,-0.002975,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
758555,2019-08-19 15:24:47,2019-08-19 15:24:47,40,51,0,0.044009,-0.087658,-0.002955,-0.007720,-0.002975,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1030464,2019-08-19 14:05:15,2019-08-19 14:05:15,40,51,0,0.596542,-0.087658,-0.002955,-0.007720,-0.002975,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1101613,2019-08-19 15:13:06,2019-08-19 15:13:06,40,51,0,0.044009,-0.087658,-0.002955,-0.007720,-0.002975,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1094820,2019-08-19 13:19:03,2019-08-19 13:19:03,40,51,0,0.906735,-0.087658,-0.002955,-0.007720,-0.002975,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [13]:
# 0 is backdoor, 1 is CommInj, ...
encoder.classes_

array(['Backdoor', 'CommInj', 'DoS', 'Reconn', 'normal'], dtype=object)

In [14]:
# create edges (in the form of an edge list first) and edge features
edge_list_train = torch.tensor(
    df_train[['SrcAddr', 'DstAddr']].to_numpy(),
    dtype=torch.long
)
edge_features_train = torch.tensor(
    df_train.drop(columns=TO_DROP).to_numpy(),
    dtype=torch.float64
)
y_train = torch.tensor(
    df_train[TARGET].to_numpy(),
    dtype=torch.long
)


edge_list_val = torch.tensor(
    df_val[['SrcAddr', 'DstAddr']].to_numpy(),
    dtype=torch.long
)
edge_features_val = torch.tensor(
    df_val.drop(columns=TO_DROP).to_numpy(),
    dtype=torch.float64
)
y_val = torch.tensor(
    df_val[TARGET].to_numpy(),
    dtype=torch.long
)


edge_list_test = torch.tensor(
    df_test[['SrcAddr', 'DstAddr']].to_numpy(),
    dtype=torch.long
)
edge_features_test = torch.tensor(
    df_test.drop(columns=TO_DROP).to_numpy(),
    dtype=torch.float64
)
y_test = torch.tensor(
    df_test[TARGET].to_numpy(),
    dtype=torch.long
)

# TODO: see if there's a way to include time into the graphs as well

In [15]:
# NOTE: we did not create any node features yet, so use all ones at the moment
edge_features_train = edge_features_train.float()
train_data = Data(
    edge_index=edge_list_train.t().contiguous(),
    edge_attr=edge_features_train.float(),
    edge_label=y_train,
    num_classes=len(y_train.unique()),
    x=torch.tensor(
        [is_private_ip[i] for i in range(len(ip_addr_mapping))],
        dtype=torch.float32
    ).unsqueeze(1)
)

edge_features_val = edge_features_val.float()
val_data = Data(
    edge_index=edge_list_val.t().contiguous(),
    edge_attr=edge_features_val.float(),
    edge_label=y_val,
    num_classes=len(y_val.unique()),
    x=torch.tensor(
        [is_private_ip[i] for i in range(len(ip_addr_mapping))],
        dtype=torch.float32
    ).unsqueeze(1)
)

edge_features_test = edge_features_test.float()
test_data = Data(
    edge_index=edge_list_test.t().contiguous(),
    edge_attr=edge_features_test.float(),
    edge_label=y_test,
    num_classes=len(y_test.unique()),
    x=torch.tensor(
        [is_private_ip[i] for i in range(len(ip_addr_mapping))],
        dtype=torch.float32
    ).unsqueeze(1)
)

train_data.validate(raise_on_error=True)
val_data.validate(raise_on_error=True)
test_data.validate(raise_on_error=True)

True

### Model definition

In [16]:
class GCN(nn.Module):
    def __init__(self, num_edge_features, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels, hidden_channels, add_self_loops=True)
        self.conv2 = GCNConv(hidden_channels, hidden_channels, add_self_loops=True)
        self.conv3 = GCNConv(hidden_channels, hidden_channels, add_self_loops=True)

        self.classifier = nn.Linear(hidden_channels * 2 + num_edge_features, out_channels)

        # xavier glorot initialization
        self._init_parameters()

    def forward(self, x: Tensor, edge_index: Tensor, edge_attr: Tensor, edge_label_index: Tensor, target_edge_attr: Tensor) -> Tensor:
        # x: Node feature matrix of shape [num_nodes, in_channels]
        # edge_index: Graph connectivity matrix of shape [2, num_edges]
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        # skip connection applied here
        identity = x
        x = self.conv2(x, edge_index)
        x = F.relu(x + identity)
        x = F.dropout(x, p=0.1, training=self.training)

        # layer 3
        x = self.conv3(x, edge_index)

        # edge classification task
        # get edge embeddings out from the node embeddings
        row, col = edge_label_index
        edge_src_emb = x[row]  # Shape: [num_edges, hidden_channels]
        edge_dst_emb = x[col]  # Shape: [num_edges, hidden_channels]
        edge_features = torch.cat([edge_src_emb, edge_dst_emb, target_edge_attr], dim=-1)

        # classifier
        x = self.classifier(edge_features)

        return x

    def _init_parameters(self):
        # xavier glorot init by default for this method
        self.conv1.reset_parameters()
        self.conv2.reset_parameters()
        self.conv3.reset_parameters()

        nn.init.xavier_normal_(self.classifier.weight)
        if self.classifier.bias is not None:
            nn.init.zeros_(self.classifier.bias)


model = GCN(
    num_edge_features=train_data.num_edge_features,
    in_channels=train_data.num_features,
    hidden_channels=32,
    out_channels=train_data.num_classes
)

In [17]:
train_dataloader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=[15, 15],
    batch_size=256,
    edge_label_index=train_data.edge_index,
    edge_label=train_data.edge_label,
    shuffle=True,
)

val_dataloader = LinkNeighborLoader(
    data=val_data,
    num_neighbors=[15, 15],
    batch_size=256,
    edge_label_index=val_data.edge_index,
    edge_label=val_data.edge_label,
    shuffle=True,
)

test_dataloader = LinkNeighborLoader(
    data=test_data,
    num_neighbors=[15, 15],
    batch_size=256,
    edge_label_index=test_data.edge_index,
    edge_label=test_data.edge_label,
    shuffle=True,
)

/usr/local/lib/python3.12/dist-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# set class weights for loss function
class_weights = compute_class_weight(class_weight='balanced', y=y_train.numpy(), classes=np.unique(y_train.numpy()))
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.to(device)

GCN(
  (conv1): GCNConv(1, 32)
  (conv2): GCNConv(32, 32)
  (conv3): GCNConv(32, 32)
  (classifier): Linear(in_features=120, out_features=5, bias=True)
)

In [19]:
def train(model, edge_features, dataloader, loss_fn, optimizer, device, num_classes=5):
    model.train()

    running_loss = 0.0

    all_preds = []
    all_labels = []
    all_probs = []

    edge_features = edge_features.to(device)

    for batch in dataloader:
        batch = batch.to(device)

        optimizer.zero_grad()

        target_edge_attr = edge_features[batch.input_id]
        # forward pass
        outputs = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            target_edge_attr
        )

        # outputs: [num_target_edges, num_classes]
        # batch.edge_label: [num_target_edges]
        loss = loss_fn(outputs, batch.edge_label)
        # backpropagation
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Probabilities
        probs = torch.softmax(outputs, dim=1)
        # Predicted class
        preds = outputs.argmax(dim=1)

        # Move to CPU for sklearn
        all_probs.append(probs.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(batch.edge_label.detach().cpu().numpy())

    # Combine all batches
    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    # Average loss
    avg_loss = running_loss / len(dataloader)

    # Classification metrics
    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_precision = precision_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    # Multiclass PR-AUC
    one_hot_labels = np.eye(num_classes)[all_labels]

    pr_auc = average_precision_score(
        one_hot_labels,
        all_probs,
        average="macro"
    )

    return {
        "loss": avg_loss,
        "f1": macro_f1,
        "precision": macro_precision,
        "recall": macro_recall,
        "pr_auc": pr_auc
    }

def evaluate(model, edge_features, dataloader, loss_fn, optimizer, device, num_classes=5):
    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []
    all_probs = []

    edge_features = edge_features.to(device)

    with torch.no_grad():

        for batch in dataloader:
            batch = batch.to(device)

            # Forward pass
            target_edge_attr = edge_features[batch.input_id]
            outputs = model(
                batch.x,
                batch.edge_index,
                batch.edge_attr,
                batch.edge_label_index,
                target_edge_attr
            )

            # Loss
            loss = loss_fn(outputs, batch.edge_label)
            running_loss += loss.item()

            # Probabilities
            probs = torch.softmax(outputs, dim=1)

            # Predicted class
            preds = outputs.argmax(dim=1)

            # Store for metrics
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(batch.edge_label.cpu().numpy())

    # Combine batches
    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    # Average loss
    avg_loss = running_loss / len(dataloader)

    # Metrics
    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_precision = precision_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    # Multiclass PR-AUC
    one_hot_labels = np.eye(num_classes)[all_labels]

    pr_auc = average_precision_score(
        one_hot_labels,
        all_probs,
        average="macro"
    )

    return {
        "loss": avg_loss,
        "f1": macro_f1,
        "precision": macro_precision,
        "recall": macro_recall,
        "pr_auc": pr_auc
    }

In [20]:
NUM_EPOCHS = 30
PATIENCE = 5

best_val_loss = float("inf")
epochs_without_improvement = 0
best_model_state = None

# TODO: maybe can try 1 pass the whole graph in but not sure if it will fit in memory or not
# TODO: add saving model after every x epochs, try changing learning rate or add learning rate scheduler
# TODO: run notebook again without the preprocessing code once preprocessing is done
# TODO: add plots of confusion matrix, etc. once done training
# TODO: train on both train and val sample for X epochs then evaluate on test data

for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = train(model, edge_features_train, train_dataloader, loss_fn, optimizer, device)
    val_metrics = evaluate(model, edge_features_val, val_dataloader, loss_fn, optimizer, device)

    print(f"Epoch {epoch}: ")
    print(
        f"Train loss = {train_metrics['loss']}, "
        f"Train f1: {train_metrics['f1']}, "
        f"Train precision: {train_metrics['precision']}, "
        f"Train recall: {train_metrics['recall']}, "
        f"Train PR-AUC: {train_metrics['pr_auc']}"
    )
    print(
        f"Validation loss = {val_metrics['loss']}, "
        f"Validation f1: {val_metrics['f1']}, "
        f"Validation precision: {val_metrics['precision']}, "
        f"Validation recall: {val_metrics['recall']}, "
        f"Validation PR-AUC: {val_metrics['pr_auc']}"
    )

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        epochs_without_improvement = 0
        # save the best model weights
        torch.save(model.state_dict(), 'model.pt')

        print("Validation loss improved. Saving model.")
    else:
        epochs_without_improvement += 1
        print(
            f"No improvement for "
            f"{epochs_without_improvement}/{PATIENCE} epochs."
        )
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break
    print("-" * 100)

Epoch 1: 
Train loss = 0.20335275757499316, Train f1: 0.549409528364449, Train precision: 0.4844744396870766, Train recall: 0.8775441490704198, Train PR-AUC: 0.8243342606864139
Validation loss = 0.054773679649784794, Validation f1: 0.6271559973711048, Validation precision: 0.5878216323776698, Validation recall: 0.9384405010768615, Validation PR-AUC: 0.891906358093727
Validation loss improved. Saving model.
----------------------------------------------------------------------------------------------------
Epoch 2: 
Train loss = 0.08563348954583143, Train f1: 0.6416884555169917, Train precision: 0.5957049092842701, Train recall: 0.9265104199676217, Train PR-AUC: 0.8689589792204655
Validation loss = 0.0474014834842936, Validation f1: 0.7257420303678658, Validation precision: 0.6735436391085052, Validation recall: 0.9670475486019473, Validation PR-AUC: 0.8996367501003325
Validation loss improved. Saving model.
-------------------------------------------------------------------------------